# 02 - Optimization Algorithms from Scratch

**Goal:** Implement every major optimizer used in deep learning from scratch using only NumPy. Visualize their behavior on standard test surfaces.

**Optimizers implemented:**
- SGD (vanilla)
- SGD + Momentum
- Nesterov Accelerated Gradient (NAG)
- AdaGrad
- RMSProp
- Adam
- AdamW (decoupled weight decay)

---

### The optimization problem in ML

We minimize $L(\theta) = \frac{1}{N} \sum_{i=1}^N \ell(f_\theta(x_i), y_i)$ where we only have access to stochastic gradient estimates $g_t \approx \nabla L(\theta_t)$ from mini-batches.

All optimizers below are variations on: $\theta_{t+1} = \theta_t - \eta \cdot \text{update}(g_t, \text{history})$

**Resources:**
- Ruder, "An overview of gradient descent optimization algorithms" (2016)
- Loshchilov & Hutter, "Decoupled Weight Decay Regularization" (AdamW, 2019)
- Zhang et al., "Which Algorithmic Choices Matter..." (2019)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

## Test Surfaces

Standard benchmark functions for optimization. These have known global minima and challenging landscapes.

**Rosenbrock:** $f(x,y) = (a-x)^2 + b(y-x^2)^2$, minimum at $(a, a^2)$. Narrow curved valley -- tests ability to navigate ill-conditioned landscapes.

**Beale:** $f(x,y) = (1.5 - x + xy)^2 + (2.25 - x + xy^2)^2 + (2.625 - x + xy^3)^2$, minimum at $(3, 0.5)$. Flat regions and sharp ridges.

In [ ]:
def rosenbrock(xy):
    """Rosenbrock function. Minimum at (1, 1) with value 0."""
    x, y = xy[0], xy[1]
    return (1 - x)**2 + 100 * (y - x**2)**2

def rosenbrock_grad(xy):
    x, y = xy[0], xy[1]
    dx = -2 * (1 - x) + 200 * (y - x**2) * (-2 * x)
    dy = 200 * (y - x**2)
    return np.array([dx, dy])

def beale(xy):
    """Beale function. Minimum at (3, 0.5) with value 0."""
    x, y = xy[0], xy[1]
    return ((1.5 - x + x*y)**2 + 
            (2.25 - x + x*y**2)**2 + 
            (2.625 - x + x*y**3)**2)

def beale_grad(xy):
    x, y = xy[0], xy[1]
    t1 = 1.5 - x + x*y
    t2 = 2.25 - x + x*y**2
    t3 = 2.625 - x + x*y**3
    dx = 2*t1*(y - 1) + 2*t2*(y**2 - 1) + 2*t3*(y**3 - 1)
    dy = 2*t1*x + 2*t2*(2*x*y) + 2*t3*(3*x*y**2)
    return np.array([dx, dy])

In [ ]:
# Visualize both surfaces
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rosenbrock
x = np.linspace(-2, 2, 200)
y = np.linspace(-1, 3, 200)
X, Y = np.meshgrid(x, y)
Z = (1 - X)**2 + 100 * (Y - X**2)**2
axes[0].contour(X, Y, Z, levels=np.logspace(-1, 3.5, 30), cmap='viridis')
axes[0].plot(1, 1, 'r*', markersize=15, label='Minimum (1,1)')
axes[0].set_title('Rosenbrock Function')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].legend()

# Beale
x = np.linspace(-1, 4.5, 200)
y = np.linspace(-1, 2, 200)
X, Y = np.meshgrid(x, y)
Z = ((1.5 - X + X*Y)**2 + (2.25 - X + X*Y**2)**2 + (2.625 - X + X*Y**3)**2)
axes[1].contour(X, Y, Z, levels=np.logspace(-1, 4, 30), cmap='viridis')
axes[1].plot(3, 0.5, 'r*', markersize=15, label='Minimum (3, 0.5)')
axes[1].set_title('Beale Function')
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
axes[1].legend()

plt.tight_layout()
plt.show()

## Optimizer Implementations

Each optimizer follows the same interface:
- `__init__(params, lr, ...)` -- stores hyperparams and initializes state
- `step(grads)` -- updates params given gradients

All updates are in-place on the `params` array.

In [ ]:
class SGD:
    """Vanilla stochastic gradient descent.
    
    Update: theta <- theta - lr * grad
    
    Simple but slow on ill-conditioned problems. Oscillates in 
    high-curvature directions.
    """
    def __init__(self, params, lr=0.01):
        self.params = params
        self.lr = lr
    
    def step(self, grads):
        self.params -= self.lr * grads


class SGDMomentum:
    """SGD with momentum.
    
    v_t = beta * v_{t-1} + grad_t
    theta <- theta - lr * v_t
    
    Momentum smooths gradients over time, damping oscillations and
    accelerating in consistent gradient directions.
    Typical beta: 0.9
    """
    def __init__(self, params, lr=0.01, beta=0.9):
        self.params = params
        self.lr = lr
        self.beta = beta
        self.v = np.zeros_like(params)
    
    def step(self, grads):
        self.v = self.beta * self.v + grads
        self.params -= self.lr * self.v


class NesterovMomentum:
    """Nesterov Accelerated Gradient (NAG).
    
    Look ahead: compute gradient at the anticipated next position.
    v_t = beta * v_{t-1} + grad(theta - lr * beta * v_{t-1})
    theta <- theta - lr * v_t
    
    In practice (equivalent reformulation for implementation):
    v_t = beta * v_{t-1} + grad_t
    theta <- theta - lr * (beta * v_t + grad_t)
    
    Nesterov momentum "looks ahead" and corrects, leading to faster
    convergence than classical momentum.
    """
    def __init__(self, params, lr=0.01, beta=0.9):
        self.params = params
        self.lr = lr
        self.beta = beta
        self.v = np.zeros_like(params)
    
    def step(self, grads):
        self.v = self.beta * self.v + grads
        self.params -= self.lr * (self.beta * self.v + grads)


class AdaGrad:
    """Adaptive Gradient.
    
    G_t = G_{t-1} + grad_t^2  (element-wise)
    theta <- theta - lr * grad_t / (sqrt(G_t) + eps)
    
    Adapts learning rate per-parameter. Parameters with large historical
    gradients get smaller effective LR. Good for sparse features.
    
    Problem: G_t only grows, so LR monotonically decreases -> premature stopping.
    """
    def __init__(self, params, lr=0.01, eps=1e-8):
        self.params = params
        self.lr = lr
        self.eps = eps
        self.G = np.zeros_like(params)
    
    def step(self, grads):
        self.G += grads ** 2
        self.params -= self.lr * grads / (np.sqrt(self.G) + self.eps)


class RMSProp:
    """Root Mean Square Propagation (Hinton, unpublished).
    
    v_t = beta * v_{t-1} + (1 - beta) * grad_t^2
    theta <- theta - lr * grad_t / (sqrt(v_t) + eps)
    
    Fixes AdaGrad's monotonically decreasing LR by using exponential
    moving average of squared gradients instead of sum.
    Typical beta: 0.99
    """
    def __init__(self, params, lr=0.001, beta=0.99, eps=1e-8):
        self.params = params
        self.lr = lr
        self.beta = beta
        self.eps = eps
        self.v = np.zeros_like(params)
    
    def step(self, grads):
        self.v = self.beta * self.v + (1 - self.beta) * grads ** 2
        self.params -= self.lr * grads / (np.sqrt(self.v) + self.eps)


class Adam:
    """Adaptive Moment Estimation (Kingma & Ba, 2015).
    
    m_t = beta1 * m_{t-1} + (1 - beta1) * g_t        (first moment / mean)
    v_t = beta2 * v_{t-1} + (1 - beta2) * g_t^2      (second moment / variance)
    m_hat = m_t / (1 - beta1^t)                        (bias correction)
    v_hat = v_t / (1 - beta2^t)                        (bias correction)
    theta <- theta - lr * m_hat / (sqrt(v_hat) + eps)
    
    Combines momentum (first moment) with adaptive LR (second moment).
    Bias correction is critical for early steps when m,v are near zero.
    Typical: beta1=0.9, beta2=0.999, lr=1e-3
    """
    def __init__(self, params, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params = params
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.m = np.zeros_like(params)
        self.v = np.zeros_like(params)
        self.t = 0
    
    def step(self, grads):
        self.t += 1
        self.m = self.beta1 * self.m + (1 - self.beta1) * grads
        self.v = self.beta2 * self.v + (1 - self.beta2) * grads ** 2
        m_hat = self.m / (1 - self.beta1 ** self.t)
        v_hat = self.v / (1 - self.beta2 ** self.t)
        self.params -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)


class AdamW:
    """Adam with decoupled weight decay (Loshchilov & Hutter, 2019).
    
    The key insight: L2 regularization != weight decay when using Adam.
    
    Standard Adam + L2: grad += lambda * theta, then Adam update
      -> The regularization gradient gets scaled by the adaptive LR,
         which means large-gradient params get LESS regularization.
    
    AdamW: Do the Adam update normally, then separately:
      theta <- theta - lr * wd * theta
      -> Weight decay is applied uniformly, not scaled by Adam's moments.
    
    This is what's actually used in practice for training transformers.
    """
    def __init__(self, params, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.01):
        self.params = params
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.wd = weight_decay
        self.m = np.zeros_like(params)
        self.v = np.zeros_like(params)
        self.t = 0
    
    def step(self, grads):
        self.t += 1
        self.m = self.beta1 * self.m + (1 - self.beta1) * grads
        self.v = self.beta2 * self.v + (1 - self.beta2) * grads ** 2
        m_hat = self.m / (1 - self.beta1 ** self.t)
        v_hat = self.v / (1 - self.beta2 ** self.t)
        # Adam step
        self.params -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)
        # Decoupled weight decay
        self.params -= self.lr * self.wd * self.params

## Visualize Optimization Trajectories

Run each optimizer from the same starting point and plot the path it takes through the loss surface.

In [ ]:
def run_optimizer(OptimizerClass, func, grad_func, start, n_steps, **kwargs):
    """Run an optimizer and return the trajectory and loss history."""
    params = np.array(start, dtype=np.float64)
    opt = OptimizerClass(params, **kwargs)
    trajectory = [params.copy()]
    losses = [func(params)]
    
    for _ in range(n_steps):
        g = grad_func(params)
        # Clip gradients to prevent divergence on steep surfaces
        g = np.clip(g, -100, 100)
        opt.step(g)
        trajectory.append(params.copy())
        losses.append(func(params))
    
    return np.array(trajectory), np.array(losses)

In [ ]:
# Run all optimizers on Rosenbrock
start = [-1.0, 1.5]
n_steps = 5000

optimizers_config = {
    'SGD':       (SGD,             {'lr': 0.0005}),
    'Momentum':  (SGDMomentum,     {'lr': 0.0003, 'beta': 0.9}),
    'Nesterov':  (NesterovMomentum,{'lr': 0.0002, 'beta': 0.9}),
    'AdaGrad':   (AdaGrad,         {'lr': 0.1}),
    'RMSProp':   (RMSProp,         {'lr': 0.003, 'beta': 0.9}),
    'Adam':      (Adam,            {'lr': 0.01}),
    'AdamW':     (AdamW,           {'lr': 0.01, 'weight_decay': 0.0}),  # no WD for fair comparison
}

results = {}
for name, (OptClass, kwargs) in optimizers_config.items():
    traj, losses = run_optimizer(OptClass, rosenbrock, rosenbrock_grad, start, n_steps, **kwargs)
    results[name] = (traj, losses)
    print(f"{name:12s} | Final loss: {losses[-1]:12.6f} | Final pos: ({traj[-1,0]:.4f}, {traj[-1,1]:.4f})")

In [ ]:
# Plot trajectories on Rosenbrock contour
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Contour plot with trajectories
ax = axes[0]
x = np.linspace(-2, 2, 300)
y = np.linspace(-1, 3, 300)
X, Y = np.meshgrid(x, y)
Z = (1 - X)**2 + 100 * (Y - X**2)**2
ax.contour(X, Y, Z, levels=np.logspace(-1, 3.5, 25), cmap='Greys', alpha=0.5)

colors = plt.cm.Set1(np.linspace(0, 1, len(results)))
for (name, (traj, _)), color in zip(results.items(), colors):
    # Plot every 50th point to avoid clutter
    ax.plot(traj[::50, 0], traj[::50, 1], '-o', color=color, label=name, 
            markersize=2, linewidth=1, alpha=0.8)

ax.plot(1, 1, 'r*', markersize=15, zorder=10)
ax.plot(start[0], start[1], 'ks', markersize=8, zorder=10)
ax.set_title('Optimization Trajectories on Rosenbrock')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.legend(fontsize=8, loc='upper left')
ax.set_xlim(-2, 2)
ax.set_ylim(-1, 3)

# Loss curves
ax = axes[1]
for (name, (_, losses)), color in zip(results.items(), colors):
    ax.semilogy(losses, color=color, label=name, linewidth=1.5)
ax.set_title('Convergence Comparison')
ax.set_xlabel('Step')
ax.set_ylabel('Loss (log scale)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Trajectories on Beale Function

In [ ]:
start_beale = [0.0, 0.0]
n_steps_beale = 3000

beale_config = {
    'SGD':       (SGD,             {'lr': 0.00005}),
    'Momentum':  (SGDMomentum,     {'lr': 0.00003, 'beta': 0.9}),
    'Nesterov':  (NesterovMomentum,{'lr': 0.00002, 'beta': 0.9}),
    'AdaGrad':   (AdaGrad,         {'lr': 0.1}),
    'RMSProp':   (RMSProp,         {'lr': 0.005, 'beta': 0.9}),
    'Adam':      (Adam,            {'lr': 0.05}),
    'AdamW':     (AdamW,           {'lr': 0.05, 'weight_decay': 0.0}),
}

results_beale = {}
for name, (OptClass, kwargs) in beale_config.items():
    traj, losses = run_optimizer(OptClass, beale, beale_grad, start_beale, n_steps_beale, **kwargs)
    results_beale[name] = (traj, losses)
    print(f"{name:12s} | Final loss: {losses[-1]:12.6f} | Final pos: ({traj[-1,0]:.4f}, {traj[-1,1]:.4f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
x = np.linspace(-1, 4.5, 300)
y = np.linspace(-1, 2, 300)
X, Y = np.meshgrid(x, y)
Z = ((1.5 - X + X*Y)**2 + (2.25 - X + X*Y**2)**2 + (2.625 - X + X*Y**3)**2)
ax.contour(X, Y, Z, levels=np.logspace(-1, 4, 25), cmap='Greys', alpha=0.5)

colors = plt.cm.Set1(np.linspace(0, 1, len(results_beale)))
for (name, (traj, _)), color in zip(results_beale.items(), colors):
    ax.plot(traj[::30, 0], traj[::30, 1], '-o', color=color, label=name,
            markersize=2, linewidth=1, alpha=0.8)

ax.plot(3, 0.5, 'r*', markersize=15, zorder=10)
ax.plot(start_beale[0], start_beale[1], 'ks', markersize=8, zorder=10)
ax.set_title('Optimization Trajectories on Beale')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.legend(fontsize=8)

ax = axes[1]
for (name, (_, losses)), color in zip(results_beale.items(), colors):
    ax.semilogy(losses, color=color, label=name, linewidth=1.5)
ax.set_title('Convergence on Beale')
ax.set_xlabel('Step')
ax.set_ylabel('Loss (log scale)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Convergence on a Toy ML Problem

Optimize a quadratic loss with different condition numbers to see how each optimizer handles ill-conditioning.

In [ ]:
def make_quadratic(condition_number, dim=20):
    """Create a quadratic f(x) = 0.5 * x^T A x with given condition number."""
    np.random.seed(42)
    # Create a random positive-definite matrix with specified condition number
    Q, _ = np.linalg.qr(np.random.randn(dim, dim))
    eigenvalues = np.logspace(0, np.log10(condition_number), dim)
    A = Q @ np.diag(eigenvalues) @ Q.T
    
    def loss(x):
        return 0.5 * x @ A @ x
    
    def grad(x):
        return A @ x
    
    return loss, grad, A


# Well-conditioned vs ill-conditioned
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_idx, cond in enumerate([10, 1000]):
    loss_fn, grad_fn, A = make_quadratic(cond, dim=20)
    x0 = np.random.randn(20)
    n_steps = 500
    
    configs = {
        'SGD':      (SGD,             {'lr': 0.5 / cond}),
        'Momentum': (SGDMomentum,     {'lr': 0.5 / cond, 'beta': 0.9}),
        'Nesterov': (NesterovMomentum,{'lr': 0.3 / cond, 'beta': 0.9}),
        'AdaGrad':  (AdaGrad,         {'lr': 0.5}),
        'RMSProp':  (RMSProp,         {'lr': 0.01, 'beta': 0.9}),
        'Adam':     (Adam,            {'lr': 0.05}),
    }
    
    ax = axes[ax_idx]
    colors = plt.cm.Set1(np.linspace(0, 1, len(configs)))
    for (name, (OptClass, kwargs)), color in zip(configs.items(), colors):
        params = x0.copy()
        opt = OptClass(params, **kwargs)
        loss_history = [loss_fn(params)]
        for _ in range(n_steps):
            g = grad_fn(params)
            g = np.clip(g, -100, 100)
            opt.step(g)
            loss_history.append(loss_fn(params))
        ax.semilogy(loss_history, color=color, label=name, linewidth=1.5)
    
    ax.set_title(f'Quadratic (condition number = {cond})')
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Demonstrating the AdamW Difference

Show why decoupled weight decay matters. With L2 regularization in Adam, the regularization gradient gets scaled by the adaptive learning rate, making it uneven across parameters.

In [ ]:
class AdamL2:
    """Adam with L2 regularization (the wrong way to do weight decay in Adam).
    
    L2 adds lambda*theta to the gradient BEFORE the Adam update,
    so it gets scaled by the adaptive learning rate.
    """
    def __init__(self, params, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8, l2_lambda=0.01):
        self.params = params
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.l2_lambda = l2_lambda
        self.m = np.zeros_like(params)
        self.v = np.zeros_like(params)
        self.t = 0
    
    def step(self, grads):
        self.t += 1
        # L2: add regularization to gradient (coupled)
        grads = grads + self.l2_lambda * self.params
        self.m = self.beta1 * self.m + (1 - self.beta1) * grads
        self.v = self.beta2 * self.v + (1 - self.beta2) * grads ** 2
        m_hat = self.m / (1 - self.beta1 ** self.t)
        v_hat = self.v / (1 - self.beta2 ** self.t)
        self.params -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)


# Compare on a problem where regularization matters
np.random.seed(42)
dim = 50
X = np.random.randn(100, dim)
true_w = np.zeros(dim)
true_w[:5] = np.array([1, -2, 3, -1, 0.5])  # sparse: only 5 non-zero
y = X @ true_w + 0.1 * np.random.randn(100)

def mse_loss(w):
    return np.mean((X @ w - y) ** 2)

def mse_grad(w):
    return 2 * X.T @ (X @ w - y) / len(y)

# Train with AdamW vs Adam+L2
n_steps = 2000
wd = 0.05

w_adamw = np.random.randn(dim) * 0.1
opt_adamw = AdamW(w_adamw, lr=0.01, weight_decay=wd)
losses_adamw = []

w_adaml2 = w_adamw.copy()  # same init
opt_adaml2 = AdamL2(w_adaml2, lr=0.01, l2_lambda=wd)
losses_adaml2 = []

for _ in range(n_steps):
    g = mse_grad(w_adamw)
    opt_adamw.step(g)
    losses_adamw.append(mse_loss(w_adamw))
    
    g = mse_grad(w_adaml2)
    opt_adaml2.step(g)
    losses_adaml2.append(mse_loss(w_adaml2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].semilogy(losses_adamw, label='AdamW', linewidth=2)
axes[0].semilogy(losses_adaml2, label='Adam + L2', linewidth=2)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('AdamW vs Adam+L2: Loss Curves')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Compare learned weights
axes[1].bar(np.arange(dim) - 0.15, w_adamw, 0.3, label='AdamW', alpha=0.7)
axes[1].bar(np.arange(dim) + 0.15, w_adaml2, 0.3, label='Adam+L2', alpha=0.7)
axes[1].bar(np.arange(dim), true_w, 0.1, label='True', color='black', alpha=0.5)
axes[1].set_xlabel('Weight index')
axes[1].set_ylabel('Weight value')
axes[1].set_title('Learned Weights (first 15 shown)')
axes[1].set_xlim(-1, 15)
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"AdamW  weight norm: {np.linalg.norm(w_adamw):.4f}")
print(f"AdamL2 weight norm: {np.linalg.norm(w_adaml2):.4f}")
print(f"True   weight norm: {np.linalg.norm(true_w):.4f}")

## Momentum Intuition: Why It Helps

On an elongated loss surface, SGD oscillates across the narrow direction while making slow progress along the valley. Momentum averages these oscillations out.

In [ ]:
# Elongated quadratic: f(x,y) = 0.5*(x^2 + 50*y^2)
def elongated(xy):
    return 0.5 * (xy[0]**2 + 50 * xy[1]**2)

def elongated_grad(xy):
    return np.array([xy[0], 50 * xy[1]])

start = [7.0, 2.0]
n = 200

traj_sgd, _ = run_optimizer(SGD, elongated, elongated_grad, start, n, lr=0.02)
traj_mom, _ = run_optimizer(SGDMomentum, elongated, elongated_grad, start, n, lr=0.02, beta=0.9)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.linspace(-8, 8, 200)
y = np.linspace(-3, 3, 200)
X, Y = np.meshgrid(x, y)
Z = 0.5 * (X**2 + 50 * Y**2)
ax.contour(X, Y, Z, levels=20, cmap='Greys', alpha=0.5)

ax.plot(traj_sgd[:, 0], traj_sgd[:, 1], 'b-o', markersize=2, linewidth=0.8, label='SGD', alpha=0.7)
ax.plot(traj_mom[:, 0], traj_mom[:, 1], 'r-o', markersize=2, linewidth=0.8, label='Momentum', alpha=0.7)
ax.plot(0, 0, 'k*', markersize=15)
ax.set_title('SGD vs Momentum on Elongated Quadratic\n(SGD oscillates, Momentum smooths the path)')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.legend()
plt.tight_layout()
plt.show()

---

## Quick Reference

### Typical Hyperparameter Ranges

| Optimizer | Learning Rate | Other |
|-----------|--------------|-------|
| SGD | 0.01 - 0.1 | -- |
| SGD+Momentum | 0.01 - 0.1 | beta=0.9 |
| Adam | 1e-4 - 1e-3 | beta1=0.9, beta2=0.999 |
| AdamW | 1e-4 - 1e-3 | beta1=0.9, beta2=0.999, wd=0.01-0.1 |

### When to Use Which

| Situation | Optimizer |
|-----------|----------|
| Default for transformers/LLMs | AdamW |
| Computer vision (ResNets etc.) | SGD+Momentum often beats Adam |
| Quick experiments / prototyping | Adam (robust defaults) |
| Sparse features / NLP embeddings | Adam or AdaGrad |
| Fine-tuning pre-trained models | AdamW with low LR + warmup |

### Interview Notes

**"Why Adam over SGD?"**
- Adam adapts LR per-parameter using running mean/variance of gradients
- Less sensitive to LR choice, faster initial convergence
- BUT: SGD+momentum often generalizes better for vision (flatter minima)

**"What's wrong with Adam?"**
1. L2 regularization is broken in Adam (use AdamW instead)
2. Can converge to sharp minima that generalize poorly
3. The proof in the original paper has a bug (AMSGrad fixes it, but rarely matters in practice)

**"Why does AdamW decouple weight decay?"**
- In Adam, the gradient is divided by sqrt(v). If you add L2 to the gradient, the L2 term also gets divided by sqrt(v), making regularization non-uniform.
- AdamW applies weight decay directly to params, bypassing the adaptive scaling.

**"Learning rate warmup -- why?"**
- Adam's bias correction handles early steps, but with large models the loss landscape is unstable early on
- Warmup gives the model time to move to a reasonable region before using the full LR

**Resources:**
- Loshchilov & Hutter, "Decoupled Weight Decay Regularization" (ICLR 2019)
- Zhang et al., "Lookahead Optimizer: k steps forward, 1 step back" (NeurIPS 2019)
- You et al., "Large Batch Training with LAMB" (ICLR 2020)